# Moscovium (Z=115) — Path to Nuclear Stability

### What this notebook covers
1. **Known isotopes** — measured half-lives for Mc-287 through Mc-290 (NUBASE2020)
2. **Island of stability** — why heavier Mc should become dramatically longer-lived
3. **Synthesis routes** — which target + beam combinations get you closest to stable Mc
4. **Wave simulation link** — how a heavier Mc would change the 3D wave interaction

---
**Labels used throughout:**
- 🟢 *Measured* — experimental data (NUBASE2020 / Oganessian et al.)
- 🟠 *Predicted* — theoretical (WS4, FRDM2012 models; ±factor-of-5 uncertainty)
- 🔭 *Speculative* — extrapolated connection to wave-interaction properties

> The Bob Lazar claim — a **stable** Mc with gravitational-wave properties — would require
> reaching roughly **Mc-299 (N=184)**, the predicted shell-closure isotope. No element 115
> isotope has ever been stable; current best is Mc-290 at 0.65 s.
> The synthesis routes below map out what it would take to get there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Dark theme consistent with wave_3d.ipynb ─────────────────────────────────
DARK = '#070712'
MID  = '#0d0d24'
EDGE = '#334488'
plt.rcParams.update({
    'figure.facecolor': DARK,  'axes.facecolor': MID,
    'text.color': '#ccd0e8',   'axes.labelcolor': '#8899cc',
    'xtick.color': '#8899cc',  'ytick.color': '#8899cc',
    'axes.edgecolor': EDGE,    'grid.color': '#1a1a3a',
    'grid.linestyle': '--',
})

# ── Moscovium (Z = 115) nuclear data ─────────────────────────────────────────
# Measured: NUBASE2020 / Oganessian et al. 2004-2015
# Predicted: Wang et al. WS4 (2014), Möller et al. FRDM2012 (2016)
# Half-lives in seconds; N = A − 115 for Z=115

MC_MEASURED = {
    287: {'t12': 0.037,  'err': 0.005, 'N': 172},
    288: {'t12': 0.164,  'err': 0.015, 'N': 173},
    289: {'t12': 0.330,  'err': 0.030, 'N': 174},
    290: {'t12': 0.650,  'err': 0.080, 'N': 175},
}

MC_PREDICTED = {
    # midpoint estimates from published model ranges (factor-of-5 uncertainty)
    291: {'t12': 2.0,    'range': (0.4,    10),   'N': 176, 'src': 'extrapolation'},
    292: {'t12': 6.0,    'range': (1.0,    30),   'N': 177, 'src': 'extrapolation'},
    293: {'t12': 18.,    'range': (3.0,   100),   'N': 178, 'src': 'FRDM2012'},
    294: {'t12': 55.,    'range': (8.0,   400),   'N': 179, 'src': 'FRDM2012'},
    295: {'t12': 160.,   'range': (20,   1500),   'N': 180, 'src': 'WS4'},
    296: {'t12': 500.,   'range': (60,   6000),   'N': 181, 'src': 'WS4'},
    297: {'t12': 1600.,  'range': (150, 25000),   'N': 182, 'src': 'WS4'},
    298: {'t12': 5000.,  'range': (400, 80000),   'N': 183, 'src': 'WS4'},
    299: {'t12': 18000., 'range': (1000, 2.0e5),  'N': 184, 'src': 'WS4 — N=184 shell'},
}

MAGIC_N = 184   # predicted next neutron magic number (island of stability)
MC_ALL  = {**MC_MEASURED, **MC_PREDICTED}

# ── Synthesis reactions ───────────────────────────────────────────────────────
# Z_target(95) + Z_beam(20) = 115 for all Am+Ca routes
# Compound A = At + Ab; products = compound − {2, 3, 4} evaporated neutrons

REACTIONS = [
    dict(
        label='Am-243 + \u2074\u2078Ca',
        At=243, Ab=48,
        target_note='standard target  (t\u2081\u2082 = 7,370 yr)',
        beam_note='stable isotope',
        performed=True,
        color='#44cc88',
        feasibility='\u2705  Performed \u2014 FLNR / GSI / LBNL',
    ),
    dict(
        label='Am-245 + \u2074\u2078Ca',
        At=245, Ab=48,
        target_note='reactor-activated Am-243  (t\u2081\u2082 = 2.05 h)',
        beam_note='stable isotope',
        performed=False,
        color='#66aaff',
        feasibility='\U0001f52c  Challenging: online Am-245 target production',
    ),
    dict(
        label='Am-247 + \u2074\u2078Ca',
        At=247, Ab=48,
        target_note='high-flux reactor  (t\u2081\u2082 = 23.5 min)',
        beam_note='stable isotope',
        performed=False,
        color='#ffaa44',
        feasibility='\u2697\ufe0f   Very hard: short-lived target + recoil separator',
    ),
    dict(
        label='Am-243 + \u2075\u2070Ca',
        At=243, Ab=50,
        target_note='standard target',
        beam_note='radioactive beam  (t\u2081\u2082 = 13.9 s)',
        performed=False,
        color='#cc88ff',
        feasibility='\U0001f3ed  ISOL radioactive beam facility required (FRIB / RIKEN)',
    ),
    dict(
        label='Am-247 + \u2075\u2070Ca  \u2605',
        At=247, Ab=50,
        target_note='reactor  (t\u2081\u2082 = 23.5 min)',
        beam_note='radioactive beam  (t\u2081\u2082 = 13.9 s)',
        performed=False,
        color='#ff6688',
        feasibility='\U0001f52d  Both species exotic \u2014 nearest to island; future facilities only',
    ),
]

# Pre-compute products for each reaction (compound − 2n, 3n, 4n)
for rx in REACTIONS:
    Ac = rx['At'] + rx['Ab']
    rx['compound_A'] = Ac
    rx['products'] = [Ac - k for k in [2, 3, 4] if (Ac - k) in MC_ALL]

# ── Helpers ───────────────────────────────────────────────────────────────────
def fmt_t12(s):
    if s < 1:      return f'{s*1000:.0f} ms'
    if s < 60:     return f'{s:.1f} s'
    if s < 3600:   return f'{s/60:.0f} min'
    if s < 86400:  return f'{s/3600:.1f} h'
    return f'{s/86400:.0f} d'

def wave_speed(A):
    """Wave-interaction speed for Mc-A in the 3D simulation.
    Interpolates between the current Moscovium preset (A=290, c=0.15)
    and the exotic preset (A=299, c=0.04).  Speculative extrapolation."""
    t = np.clip((A - 290) / (299 - 290), 0, 1)
    return 0.15 * (1 - t) + 0.04 * t

# ── Overview figure ───────────────────────────────────────────────────────────
def plot_overview():
    fig = plt.figure(figsize=(15, 9), facecolor=DARK)
    gs  = gridspec.GridSpec(
        2, 2, width_ratios=[1.4, 1], height_ratios=[1.6, 1],
        hspace=0.40, wspace=0.38, left=0.07, right=0.97, top=0.90, bottom=0.09,
    )
    ax_hl    = fig.add_subplot(gs[:, 0])   # left:  half-life trend
    ax_route = fig.add_subplot(gs[0, 1])   # top-right: route ladder
    ax_wave  = fig.add_subplot(gs[1, 1])   # bottom-right: wave speed

    # ── Panel 1: Half-life trend ──────────────────────────────────────────────
    all_A = sorted(MC_ALL.keys())
    for A in all_A:
        is_meas = A in MC_MEASURED
        d = MC_ALL[A]
        t12 = d['t12']

        # colour gradient: green (measured) → warm orange → deep red (predicted)
        if is_meas:
            col = '#44cc88'
        else:
            frac = (A - 291) / 8.0
            r = int(255 * min(1.0, 0.70 + 0.30 * frac))
            g = int(255 * max(0.0, 0.55 - 0.40 * frac))
            b = int(255 * 0.18)
            col = f'#{r:02x}{g:02x}{b:02x}'

        hatch = '' if is_meas else '//'
        ax_hl.bar(A, t12, color=col, alpha=0.88, edgecolor='#334488',
                  linewidth=0.8, hatch=hatch, zorder=3)

        # error bars
        if is_meas:
            e = d['err']
            ax_hl.errorbar(A, t12, yerr=[[e], [e]],
                           fmt='none', color='white', capsize=3, lw=1.2, zorder=4)
        else:
            lo, hi = d['range']
            ax_hl.errorbar(A, t12,
                           yerr=[[t12 - lo], [hi - t12]],
                           fmt='none', color='#aaaaaa', capsize=3, lw=1.0, zorder=4)

        # value label
        ax_hl.text(A, t12 * 1.6, fmt_t12(t12),
                   ha='center', va='bottom', fontsize=6.8,
                   color='white', rotation=65, zorder=5)

    # Annotations
    ax_hl.axvline(299, color='#ffdd44', lw=2.0, ls='--', alpha=0.85, zorder=2)
    ax_hl.text(299.25, 5e-3,
               'N = 184\nshell closure\n(island of stability)',
               color='#ffdd44', fontsize=8.5, va='bottom')
    ax_hl.axvline(290.5, color='#334488', lw=1.5, ls=':', zorder=2)
    ax_hl.text(290.65, 3e-3, '\u2190 measured | predicted \u2192',
               color='#667799', fontsize=7.5)

    ax_hl.set_yscale('log')
    ax_hl.set_xlabel('Mass number  A', fontsize=11)
    ax_hl.set_ylabel('Half-life', fontsize=11)
    ax_hl.set_title('Moscovium (Z=115) \u2014 Known & Predicted Half-Lives',
                    fontsize=12, color='#aabbff', pad=8)
    ax_hl.set_xticks(all_A)
    ax_hl.set_xlim(286, 300.5)
    ax_hl.set_ylim(5e-3, 5e6)
    ax_hl.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: fmt_t12(v)))
    ax_hl.grid(axis='y', zorder=0)

    p1 = mpatches.Patch(color='#44cc88', label='Measured (NUBASE2020)')
    p2 = mpatches.Patch(color='#dd8833', hatch='//', edgecolor='#334488',
                         label='Predicted (WS4 / FRDM2012, \xb1 factor of 5)')
    ax_hl.legend(handles=[p1, p2], fontsize=8.5, loc='upper left',
                 facecolor=MID, edgecolor=EDGE)

    # ── Panel 2: Route ladder ─────────────────────────────────────────────────
    ax_route.axhline(MAGIC_N, color='#ffdd44', lw=2.0, ls='--', alpha=0.85, zorder=5)
    ax_route.text(0.03, MAGIC_N + 0.25, 'N=184  (island of stability)',
                  color='#ffdd44', fontsize=8, transform=ax_route.get_yaxis_transform())
    ax_route.axhline(175, color='#44cc88', lw=1.0, ls=':', alpha=0.6, zorder=3)
    ax_route.text(0.03, 175.1, 'current record  Mc-290, N=175',
                  color='#44cc88', fontsize=7.5, transform=ax_route.get_yaxis_transform())

    for i, rx in enumerate(REACTIONS):
        col = rx['color']
        x   = i + 0.5
        ns  = [p - 115 for p in rx['products']]   # N = A - Z = A - 115
        if not ns:
            continue
        lo, hi = min(ns), max(ns)
        # range bar
        ax_route.plot([x, x], [lo - 0.15, hi + 0.15],
                      color=col, lw=9, alpha=0.4, solid_capstyle='round', zorder=2)
        # dots
        for n in ns:
            ax_route.scatter(x, n, color=col, s=55, zorder=4,
                             edgecolors='white', linewidths=0.9)
        # reaction label
        ax_route.text(x, lo - 1.1, rx['label'],
                      ha='center', va='top', fontsize=6.8, color=col, rotation=18)
        # gap arrow to island
        gap = MAGIC_N - hi
        ax_route.annotate(
            '', xy=(x, MAGIC_N - 0.2), xytext=(x, hi + 0.25),
            arrowprops=dict(arrowstyle='->', color=col, lw=1.3, alpha=0.45),
        )
        ax_route.text(x + 0.08, (MAGIC_N + hi) / 2,
                      f'\u03b4={gap}n', fontsize=7.5, color=col, va='center', alpha=0.9)

    ax_route.set_xlim(0, len(REACTIONS))
    ax_route.set_ylim(170, 188)
    ax_route.set_yticks(range(172, 187, 2))
    ax_route.set_xticks([])
    ax_route.set_ylabel('Neutron number  N', fontsize=10)
    ax_route.set_title('How close does each reaction\nget to the island?',
                        fontsize=10, color='#aabbff')
    ax_route.grid(axis='y', zorder=0)

    # ── Panel 3: Wave-speed extrapolation ─────────────────────────────────────
    A_range = np.arange(287, 300)
    c_vals  = [wave_speed(A) for A in A_range]
    bar_col = plt.cm.plasma(np.linspace(0.25, 0.88, len(A_range)))
    ax_wave.bar(A_range, c_vals, color=bar_col, edgecolor='#334488', lw=0.7)

    ax_wave.axhline(0.15, color='#cc44ff', lw=1.5, ls='--', alpha=0.85)
    ax_wave.text(299.3, 0.152, '\u201cMoscovium\u201d preset\n(wave_3d.ipynb)',
                 color='#cc44ff', fontsize=7, va='bottom')
    ax_wave.axhline(0.04, color='#00ffdd', lw=1.5, ls='--', alpha=0.85)
    ax_wave.text(299.3, 0.042, '\u201cExotic\u201d preset\n(wave_3d.ipynb)',
                 color='#00ffdd', fontsize=7, va='bottom')

    ax_wave.set_xlabel('Mass number  A', fontsize=10)
    ax_wave.set_ylabel('Wave speed  c / c\u2080  (\U0001f52d speculative)', fontsize=9)
    ax_wave.set_title('If we ran the wave sim with each\nMc isotope\u2026',
                       fontsize=10, color='#aabbff')
    ax_wave.set_xticks(A_range[::2])
    ax_wave.set_xlim(286, 300.5)
    ax_wave.set_ylim(0, 0.19)
    ax_wave.grid(axis='y', zorder=0)

    fig.suptitle(
        'Moscovium (Mc, Z=115) \u2014 Nuclear Stability & Synthesis Roadmap',
        fontsize=13, color='#aabbff', fontweight='bold',
    )
    plt.savefig('/tmp/mc_overview.png', dpi=140, bbox_inches='tight', facecolor=DARK)
    plt.show()


# ── Interactive synthesis explorer ───────────────────────────────────────────
def launch_explorer():

    target_opts = {
        'Am-243   (t\u2081\u2082 = 7,370 yr)   \u2014 standard target': 243,
        'Am-245   (t\u2081\u2082 = 2.05 h)    \u2014 reactor-activated': 245,
        'Am-247   (t\u2081\u2082 = 23.5 min)  \u2014 online production': 247,
    }
    beam_opts = {
        '\u2074\u2078Ca  (stable isotope)                         ': 48,
        '\u2075\u2070Ca  (t\u2081\u2082 = 13.9 s)  \u2014 radioactive beam': 50,
    }

    target_dd = widgets.Dropdown(
        options=list(target_opts.keys()),
        value=list(target_opts.keys())[0],
        description='Target:',
        style={'description_width': '70px'},
        layout=widgets.Layout(width='500px'),
    )
    beam_dd = widgets.Dropdown(
        options=list(beam_opts.keys()),
        value=list(beam_opts.keys())[0],
        description='Beam:',
        style={'description_width': '70px'},
        layout=widgets.Layout(width='500px'),
    )
    out = widgets.Output()

    def update(_=None):
        out.clear_output(wait=True)
        At = target_opts[target_dd.value]
        Ab = beam_opts[beam_dd.value]
        Ac = At + Ab
        prods = [Ac - k for k in [2, 3, 4] if (Ac - k) in MC_ALL]

        # Find matching pre-built reaction for feasibility text
        feas = ''
        for rx in REACTIONS:
            if rx['At'] == At and rx['Ab'] == Ab:
                feas = rx['feasibility']
                break
        if not feas:
            feas = '\u2753  Reaction not in database (novel combination)'

        with out:
            fig, axes = plt.subplots(
                1, 2, figsize=(12, 5),
                facecolor=DARK,
                gridspec_kw={'width_ratios': [1.4, 1]},
            )

            # ── Left: half-life bars for products ────────────────────────────
            ax = axes[0]
            for A in prods:
                d = MC_ALL[A]
                t12 = d['t12']
                is_meas = A in MC_MEASURED
                col   = '#44cc88' if is_meas else '#dd8833'
                hatch = '' if is_meas else '//'
                ax.bar(A, t12, color=col, edgecolor='#334488',
                       hatch=hatch, alpha=0.9, zorder=3, width=0.7)

                if is_meas:
                    e = d['err']
                    ax.errorbar(A, t12, yerr=[[e],[e]],
                                fmt='none', color='white', capsize=3, lw=1.2, zorder=4)
                else:
                    lo, hi = d['range']
                    ax.errorbar(A, t12,
                                yerr=[[t12-lo],[hi-t12]],
                                fmt='none', color='#aaaaaa', capsize=3, lw=1.0, zorder=4)

                N   = d['N']
                gap = MAGIC_N - N
                tag = '(measured)' if is_meas else '(predicted)'
                ax.text(A, t12 * 1.8,
                        f'{fmt_t12(t12)}\n{tag}',
                        ha='center', va='bottom', fontsize=9, color='white', zorder=5)
                ax.text(A, t12 * 0.65,
                        f'N={N}\n\u03b4={gap}\u2009n to island',
                        ha='center', va='top', fontsize=8, color='#8899cc', zorder=5)

                # Wave speed annotation
                c = wave_speed(A)
                ax.text(A, max(prods and [MC_ALL[p]['t12'] for p in prods] or [1]) * 6,
                        f'\U0001f52d wave c={c:.2f}',
                        ha='center', fontsize=7.5, color='#9988cc', zorder=5)

            ax.set_yscale('log')
            ax.set_xlabel('Product isotope  A', fontsize=11)
            ax.set_ylabel('Half-life', fontsize=11)
            tname = target_dd.value.split('\u2014')[0].strip()
            bname = beam_dd.value.split('\u2014')[0].strip()
            ax.set_title(
                f'{tname}  +  {bname}\ncompound nucleus: Mc-{Ac}',
                fontsize=10, color='#aabbff',
            )
            ax.yaxis.set_major_formatter(
                plt.FuncFormatter(lambda v, _: fmt_t12(v)))
            ax.set_xticks(prods)
            ax.grid(axis='y', zorder=0)

            p1 = mpatches.Patch(color='#44cc88', label='Measured')
            p2 = mpatches.Patch(color='#dd8833', hatch='//',
                                 edgecolor='#334488', label='Predicted')
            ax.legend(handles=[p1, p2], fontsize=9,
                      facecolor=MID, edgecolor=EDGE)

            # ── Right: N-ladder to island ─────────────────────────────────────
            ax2 = axes[1]
            ax2.set_xlim(-0.6, 0.6)
            ax2.set_ylim(170, 188)
            ax2.set_xticks([])
            ax2.set_ylabel('Neutron number  N', fontsize=11)
            ax2.set_title('Proximity to island of stability', fontsize=10, color='#aabbff')
            ax2.set_yticks(range(170, 188, 2))
            ax2.grid(axis='y', zorder=0)

            # reference lines
            ax2.axhline(MAGIC_N, color='#ffdd44', lw=2.5, ls='--', zorder=5)
            ax2.text(0.55, MAGIC_N - 0.1, 'N=184\n(island)',
                     color='#ffdd44', fontsize=9, ha='right', va='bottom')
            ax2.axhline(175, color='#44cc88', lw=1.0, ls=':', alpha=0.6)
            ax2.text(0.55, 175.1, 'Mc-290\ncurrent record',
                     color='#44cc88', fontsize=8, ha='right')

            n_pts = len(prods)
            for idx, A in enumerate(prods):
                d   = MC_ALL[A]
                N   = d['N']
                col = '#44cc88' if A in MC_MEASURED else '#dd8833'
                xp  = (idx - n_pts / 2 + 0.5) * 0.28
                ax2.scatter(xp, N, s=200, color=col, zorder=6,
                            edgecolors='white', linewidths=1.5)
                ax2.text(xp, N - 0.6, f'Mc-{A}',
                         ha='center', va='top', fontsize=9, color=col)
                ax2.annotate(
                    '', xy=(xp, MAGIC_N - 0.3), xytext=(xp, N + 0.4),
                    arrowprops=dict(arrowstyle='->', color=col, lw=1.8, alpha=0.5),
                )

            fig.suptitle(feas, fontsize=10, color='#aabbff', y=0.02)
            plt.tight_layout(rect=[0, 0.06, 1, 1])
            plt.show()

    target_dd.observe(update, names='value')
    beam_dd.observe(update, names='value')
    update()  # render immediately

    header = widgets.HTML(
        '<h3 style="color:#8899ff;font-family:monospace;margin:6px 0 2px">'
        '\u26db  Moscovium Synthesis Explorer</h3>'
        '<p style="color:#667788;font-size:12px;margin:0 0 8px">'
        'Pick a target nucleus and beam. The fusion produces a compound nucleus '
        'Mc-A that evaporates 2\u20134 neutrons, leaving the product. '
        '<b>Goal:</b> maximise N to approach N=184 (island of stability). '
        'The \U0001f52d marker shows the corresponding wave-speed preset in wave_3d.ipynb.</p>'
    )
    display(widgets.VBox([
        header,
        widgets.HBox([target_dd, beam_dd]),
        out,
    ]))


print('Setup complete. Run the next cell.')

In [ ]:
plot_overview()          # static 3-panel overview
launch_explorer()        # interactive synthesis explorer